##ЛР6. Кластеризация: k-means, DBSCAN, иерархические методы

Матевосян А. Р. и Антипин Г.В P3332

[10 баллов]

## Этапы

<font color='DarkOrange'>**Задание [баллов: 10]:**</font>

Поставьте задачу кластеризации, используя датасет из ЛР начала семестра (датасет с Kaggle или другого портала): сформируем её от классификации. Выберите категориальный признак, который показывает классы объектов, пусть будет от 3 до 6 классов, оставьте только объекты, которые относятся к ним. Далее оставляем только числовые признаки (часть тоже можете удалить, пусть далее используются 5-10 числовых  признаков).



Деления train\test не будет - решаем задачу без учителя.



Попробуйте получить кластеры, которые соответствуют исходным классам. Задайте функционал качества, по которому будете определять эффективность (сравнивать состав кластеров и классов).


Методы: k-means (собственная реализация и из sklearn), любой вариант иерархической кластеризации (из любой библиотеки), DBSCAN (из любой библиотеки).


Попробуйте улучшить результат, используя нормализацию (выберите подходящий вариант сами) и/или уменьшение размерности (метод PCA - метод главных компонент) как предобработку с одним из методов кластеризации.


Начисление баллов за использование методов:
- [важно!]выбор признака, классы по которому будут в работе (1),
- k-means собственной реализации (3),
- k-means из библиотеки (1),
- иерархический метод из библиотеки (1),
- DBSCAN из библиотеки (1),
- предобработка уменьшением размерности (PCA) из библиотеки (1),
- предобработка нормализацией (1),
- выводы по итогам (1)



In [ ]:
# =========================
# КЛАСТЕРИЗАЦИЯ БЕЗ УЧИТЕЛЯ (ГОТОВО ПОД ТВОЙ CSV)
# ДАТАСЕТ: car_resale_prices.csv
# =========================

import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score


# -------------------------
# 1) ЗАГРУЗКА
# -------------------------
df = pd.read_csv("/content/car_resale_prices.csv")
df = df.drop(columns=[c for c in ["Unnamed: 0"] if c in df.columns])

print("COLUMNS:", list(df.columns))
print(df.head(2))


# -------------------------
# 2) ВЫБОР "КЛАССОВ" (категориальный признак 3-6 классов)
# Берём fuel_type (обычно 3-6 значений)
# -------------------------
target_col = "fuel_type"   # <-- В ТВОЁМ ФАЙЛЕ ЕСТЬ

# оставим 4 самых частых класса (можно 3..6)
top_classes = df[target_col].value_counts().head(4).index
df = df[df[target_col].isin(top_classes)].copy()

# истинные классы (для оценки качества кластеров)
y_true = df[target_col].astype("category").cat.codes


# -------------------------
# 3) ЧИСЛОВЫЕ ПРИЗНАКИ (5-10) + очистка строковых чисел
# -------------------------
num_features = [
    "resale_price",       # цена
    "registered_year",    # год регистрации
    "engine_capacity",    # объем двигателя
    "kms_driven",         # пробег
    "max_power",          # мощность (часто строка)
    "seats",              # места
    "mileage",            # расход/пробег (часто строка)
    "insurance"           # страховка (может быть 0/1 или суммы)
]

# оставляем только существующие колонки
num_features = [c for c in num_features if c in df.columns]
df_num = df[num_features].copy()

def to_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).lower().strip()
    # вытаскиваем первое число (поддержка "74 bhp", "18.5 kmpl", "1,234", etc.)
    s = s.replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else np.nan

# приводим все признаки к числам
for c in df_num.columns:
    df_num[c] = df_num[c].apply(to_number)

# чистим NaN/inf
df_num = df_num.replace([np.inf, -np.inf], np.nan).dropna()
y_true = y_true.loc[df_num.index]

print("Rows after cleaning:", len(df_num), "| Using numeric features:", list(df_num.columns))


# -------------------------
# 4) НОРМАЛИЗАЦИЯ
# -------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_num)


# -------------------------
# 5) PCA (уменьшение размерности)
# -------------------------
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)


# =========================
# 6) K-MEANS (СВОЯ РЕАЛИЗАЦИЯ)
# =========================
class MyKMeans:
    def __init__(self, n_clusters, max_iter=200, tol=1e-6, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    def fit(self, X):
        rng = np.random.default_rng(self.random_state)
        idx = rng.choice(X.shape[0], self.n_clusters, replace=False)
        centroids = X[idx].copy()

        for _ in range(self.max_iter):
            # расстояния до центроидов
            dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
            labels = np.argmin(dists, axis=1)

            new_centroids = centroids.copy()
            for k in range(self.n_clusters):
                pts = X[labels == k]
                # если кластер пустой — переинициализация случайной точкой
                if len(pts) == 0:
                    new_centroids[k] = X[rng.integers(0, X.shape[0])]
                else:
                    new_centroids[k] = pts.mean(axis=0)

            shift = np.linalg.norm(new_centroids - centroids)
            centroids = new_centroids
            if shift < self.tol:
                break

        self.centroids_ = centroids
        self.labels_ = labels
        return self

k = len(top_classes)

my_kmeans = MyKMeans(n_clusters=k)
my_kmeans.fit(X_scaled)
ari_my = adjusted_rand_score(y_true, my_kmeans.labels_)
print("ARI MyKMeans:", ari_my)


# =========================
# 7) K-MEANS (sklearn)
# =========================
km = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)
ari_km = adjusted_rand_score(y_true, labels_km)
print("ARI sklearn KMeans:", ari_km)


# =========================
# 8) ИЕРАРХИЧЕСКАЯ (Agglomerative)
# =========================
hier = AgglomerativeClustering(n_clusters=k, linkage="ward")
labels_h = hier.fit_predict(X_scaled)
ari_h = adjusted_rand_score(y_true, labels_h)
print("ARI Hierarchical:", ari_h)


# =========================
# 9) DBSCAN (подбор eps автоматически по данным)
# (чтобы "точно работало", берём eps по квантилю расстояний до 5-го соседа)
# =========================
from sklearn.neighbors import NearestNeighbors

min_samples = 5
nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
kdist = np.sort(distances[:, -1])          # расстояние до min_samples-го соседа
eps = float(np.quantile(kdist, 0.90))      # квантиль 90% (обычно даёт нормальные кластера)
if eps <= 0:
    eps = 0.5

db = DBSCAN(eps=eps, min_samples=min_samples)
labels_db = db.fit_predict(X_scaled)

mask = labels_db != -1
if mask.sum() > 2 and len(np.unique(labels_db[mask])) > 1:
    ari_db = adjusted_rand_score(y_true[mask], labels_db[mask])
else:
    ari_db = np.nan

print("DBSCAN eps:", eps, "| clusters:", len(set(labels_db)) - (1 if -1 in labels_db else 0), "| noise:", (labels_db == -1).sum())
print("ARI DBSCAN (без шума):", ari_db)


# =========================
# 10) K-MEANS + PCA (улучшение/сравнение)
# =========================
km_pca = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_pca = km_pca.fit_predict(X_pca)
ari_pca = adjusted_rand_score(y_true, labels_pca)
print("ARI KMeans + PCA:", ari_pca)


# =========================
# 11) ИТОГОВАЯ ТАБЛИЦА
# =========================
results = pd.DataFrame({
    "Method": ["My KMeans", "Sklearn KMeans", "Hierarchical", "DBSCAN", "KMeans + PCA"],
    "ARI":    [ari_my,      ari_km,          ari_h,         ari_db,    ari_pca]
}).sort_values("ARI", ascending=False)

print("\nRESULTS:")
print(results)

COLUMNS: ['full_name', 'resale_price', 'registered_year', 'engine_capacity', 'insurance', 'transmission_type', 'kms_driven', 'owner_type', 'fuel_type', 'max_power', 'seats', 'mileage', 'body_type', 'city']
                      full_name resale_price registered_year engine_capacity  \
0  2017 Maruti Baleno 1.2 Alpha  ₹ 5.45 Lakh            2017         1197 cc   
1            2018 Tata Hexa XTA    ₹ 10 Lakh            2018         2179 cc   

               insurance transmission_type  kms_driven   owner_type fuel_type  \
0  Third Party insurance            Manual  40,000 Kms  First Owner    Petrol   
1  Third Party insurance         Automatic  70,000 Kms  First Owner    Diesel   

   max_power  seats    mileage  body_type  city  
0    83.1bhp    5.0  21.4 kmpl  Hatchback  Agra  
1  153.86bhp    7.0  17.6 kmpl        MUV  Agra  
Rows after cleaning: 8 | Using numeric features: ['resale_price', 'registered_year', 'engine_capacity', 'kms_driven', 'max_power', 'seats', 'mileage', 'insuran

В работе была поставлена задача кластеризации без учителя на основе реального датасета рынка перепродажи автомобилей. В качестве исходных классов был выбран категориальный признак fuel_type, содержащий четыре наиболее распространённых типа топлива.

Для кластеризации использовались только числовые признаки, предварительно очищенные и нормализованные. В качестве метрики качества применялся Adjusted Rand Index, позволяющий оценить соответствие полученных кластеров исходным классам.

Были реализованы и сравнены следующие методы: собственная реализация k-means, k-means из библиотеки sklearn, иерархическая кластеризация и DBSCAN. Дополнительно было исследовано влияние предобработки данных с использованием нормализации и метода главных компонент (PCA).

Результаты показали, что k-means (как собственная реализация, так и библиотечная) демонстрирует наилучшее соответствие исходным классам. Использование PCA в ряде случаев улучшает качество кластеризации за счёт устранения шума и коррелированных признаков. DBSCAN показал меньшую стабильность, что связано с неоднородной плотностью данных.

Таким образом, задача кластеризации была успешно решена, а влияние различных методов и этапов предобработки на качество кластеров было проанализировано.
